In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

In [3]:
# Load and preprocess data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


## Custom Model

In [5]:
def identity_block(x, filters):
    f1, f2 = filters
    shortcut = x

    x = layers.Conv2D(f1, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f2, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

In [6]:
def conv_block(x, filters, stride=2):
    f1, f2 = filters
    shortcut = layers.Conv2D(f2, (1,1), strides=stride)(x)
    shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Conv2D(f1, (3,3), strides=stride, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(f2, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

In [7]:
def ResNet18(input_shape=(32,32,3), num_classes=10):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (3,3), strides=1, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)

    x = conv_block(x, [64, 64], stride=1)
    x = identity_block(x, [64, 64])

    x = conv_block(x, [128, 128])
    x = identity_block(x, [128, 128])

    x = conv_block(x, [256, 256])
    x = identity_block(x, [256, 256])

    x = conv_block(x, [512, 512])
    x = identity_block(x, [512, 512])

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [8]:
model = ResNet18()

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=10, validation_split=0.1)

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 113s 61ms/step - accuracy: 0.4492 - loss: 1.5650 - val_accuracy: 0.5970 - val_loss: 1.1591
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 117s 52ms/step - accuracy: 0.7126 - loss: 0.8227 - val_accuracy: 0.5216 - val_loss: 1.6034
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 74s 52ms/step - accuracy: 0.7935 - loss: 0.5969 - val_accuracy: 0.7686 - val_loss: 0.6785
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.8461 - loss: 0.4487 - val_accuracy: 0.7852 - val_loss: 0.6482
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.8799 - loss: 0.3401 - val_accuracy: 0.8042 - val_loss: 0.6335
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 73s 52ms/step - accuracy: 0.9157 - loss: 0.2444 - val_accuracy: 0.8318 - val_loss: 0.5514
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 52ms/step - accuracy: 0.9403 - loss: 0.1739 - val_accuracy: 0.8160 - val_loss: 0.6122
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 53ms/step - accuracy: 0.9564

In [9]:
model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8312 - loss: 0.6932


[0.7096881866455078, 0.8269000053405762]